# 00 · Preprocessing: from the dataset to clean tables

This notebook explains how the raw dataset becomes the clean tables in `data/processed/` that the three RQ notebooks use.

The heavy steps (cloning about 3,900 repositories, replaying their history, checking imports in every commit) take many hours. Their outputs are cached in `data/intermediate/`. Each heavy step below has a cell that re-runs it, switched off by `RUN_PIPELINE = False`.

| Step | Script (`src/`) | What it does | Output (`data/intermediate/`) |
|---|---|---|---|
| 1 | `p00_select_candidates.py` | pick context files in engineered projects | `context_file_candidates.csv` |
| 2 | `p01_fetch_context_files.py` | download context files, READMEs, `.gitignore`, file trees | `fetched_files/`, `file_trees/`, `gitignore/` |
| 3 | `p02_replay_claim_history.py` | clone each project, replay every version, check each path claim | `claim_histories/` |
| 4 | `p03_claim_outcomes.py` | drop git-ignored paths, give each claim its history outcome | `claim_histories.pkl` |
| 5 | `p04_find_rule_sentences.py` + LLM | find rule sentences, LLM writes structured rules | `rule_classification/` |
| 6 | `p05_check_rules_and_commits.py` | import checker at writing, snapshot and in every later commit | `rule_checks/` |
| 7 | `p06`, `p07`, `p08` | detect agent-assisted commits (message, account, pull-request traces) | `agent_traces/`, `commit_groups_raw.csv` |
| 8 | `p09_context_file_creators.py` | agent trace on each context file's first commit | `context_file_creators.csv` |
| 9 | `r01`–`r04` | draw the samples for the manual checks, collect and merge the labels | `review/` |
| 10 | `p10`, `p11` | rule status, commit groups, who wrote broken rules | `rule_status.csv`, `commit_groups.csv`, `rule_attribution.csv` |
| 11 | `p12_export_clean_tables.py` | write the clean tables and the review sheets | `data/processed/`, `manual_review/` |

In [1]:
import os, sys; sys.path.insert(0, '../src' if os.path.basename(os.getcwd()) == 'notebooks' else 'src')
from analysis_helpers import *   # moves to the package root and loads helpers
import subprocess
RUN_PIPELINE = False   # set True to re-run a heavy step (needs network, git, and hours of time)
CLONES, CLONES_FULL = 'clones', 'clones_full'   # where the repository clones go
def run(script):
    if RUN_PIPELINE:
        subprocess.run([sys.executable, f'src/{script}'], check=True, env={**os.environ, 'CLONES': CLONES, 'CLONES_FULL': CLONES_FULL})
    else:
        print(f'skipped {script} (cached output is used)')

## Step 1 · Candidate context files

Input: `data/raw/context_files.csv` and `data/raw/repos.csv` from the Galster et al. dataset (Zenodo 10.5281/zenodo.19375881). Download them into `data/raw/`.

We keep files that agents load by default (`AGENTS.md`, `CLAUDE.md`, `.claude/CLAUDE.md`, `.github/copilot-instructions.md`, `GEMINI.md`, `.cursorrules`), in projects labelled *engineered*, with at least 20 lines, not empty and not a pointer to another file.

In [2]:
if os.path.exists('data/raw/context_files.csv'):
    subprocess.run([sys.executable, 'src/p00_select_candidates.py'], check=True)
cand = pd.read_csv('data/intermediate/context_file_candidates.csv')
print(f'{len(cand):,} candidate files in {cand.repo_name.nunique():,} projects')
cand.context_file.value_counts().rename('files').to_frame()

4,372 candidate files in 3,909 projects


,files
context_file,
AGENTS.md,1757
CLAUDE.md,1424
.github/copilot-instructions.md,1005
.claude/CLAUDE.md,91
.cursorrules,51
GEMINI.md,44


## Steps 2–4 · Path claims and their history

`p01` downloads the files. `p02` clones each project (only the commits since two days before its first context file) and replays every version of each context file and README. For each version it extracts path claims and checks them against the file tree of that commit. `p03` removes paths ignored by `.gitignore` and gives each claim its *history outcome*.

In [3]:
run('p01_fetch_context_files.py'); run('p02_replay_claim_history.py'); run('p03_claim_outcomes.py')

skipped p01_fetch_context_files.py (cached output is used)
skipped p02_replay_claim_history.py (cached output is used)
skipped p03_claim_outcomes.py (cached output is used)


**What the extractor does**, shown on the example from Fig. 1 of the paper. Inline claims are paths written as code; tree claims come from folder diagrams.

In [4]:
from claims_extractor import extract
example = '''## Project structure
- `src/server/routes/` -- HTTP handlers
- `src/server/db/` -- database access (Prisma)
- `src/client/` -- React front end

```
my-app/
├── src/
│   ├── api/
│   └── utils.ts
```
## Architecture rules
- Never import from `src/server/` in `src/client/`.
'''
pd.DataFrame(extract(example))[['path', 'kind', 'root', 'arch']]

,path,kind,root,arch
0,src/server/routes/,inline,None,True
1,src/server/db/,inline,None,True
2,src/client/,inline,None,True
3,src/server/,inline,None,True


**The funnel** from candidates to the analysed sample. A project is in the sample if its context file makes at least one path claim.

In [5]:
projects, files, claims = load('projects'), load('context_files'), load('path_claims')
funnel = pd.DataFrame([
    ('candidate files / projects', len(files), projects.repo.nunique()),
    ('projects whose history could be cloned', None, projects.history_cloned.sum()),
    ('in sample: files / projects with >= 1 path claim', files.in_sample.sum(), projects.in_sample.sum()),
], columns=['step', 'files', 'projects'])
funnel

,step,files,projects
0,candidate files / projects,4372.0,3909
1,projects whose history could be cloned,NaN,3834
2,in sample: files / projects with >= 1 path claim,3602.0,3346


**Table II (sample overview).**

In [6]:
sp, sf = projects[projects.in_sample], files[files.in_sample]
print('context files by type:'); print(sf.file.value_counts().to_string())
print('\nmain language:'); print(sp.main_language.value_counts().head(10).to_string())
print(f"\nmedian stars {sp.stars.median():.0f}, contributors {sp.contributors.median():.0f}, commits {sp.commits.median():.0f}")
print(f"median context file length {sf.lines.median():.0f} lines")
print(f"files first committed with an agent trace: {pct(sf.first_commit_has_agent_trace.mean(), 0)}")

context files by type:
file
AGENTS.md                          1463
CLAUDE.md                          1229
.github/copilot-instructions.md     780
.claude/CLAUDE.md                    78
GEMINI.md                            30
.cursorrules                         22

main language:
main_language
TypeScript    817
Python        637
Go            474
C#            247
Rust          233
JavaScript    232
Java          232
C++           197
PHP           192
C              85

median stars 530, contributors 36, commits 1842
median context file length 142 lines
files first committed with an agent trace: 25%


**The clean claim table** `data/processed/path_claims.csv` has one row per claim (context files and READMEs). The columns:

| column | meaning |
|---|---|
| `document` | `context_file` or `readme` |
| `form` | `inline` (path written as code) or `tree` (from a folder diagram) |
| `in_structure_section` | the claim sits under a heading about structure or architecture |
| `written_commit`, `written_at` | first version of the file that contains the claim |
| `valid_when_written` | the path existed at that commit |
| `in_file_at_snapshot`, `found_at_snapshot` | claim still in the file / path exists at the snapshot |
| `path_removed_at` | last commit that touched the path, if the path disappeared |
| `removed_from_file_at` | commit where the claim was removed from the file |
| `history_outcome` | `still_valid`, `stale_fixed`, `stale_unfixed`, `deleted_while_valid`, `never_valid`, `never_valid_deleted`, `became_valid` |

In [7]:
claims.head(3).T

,0,1,2
claim_id,C00000,C00001,C00002
repo,daisy/MathCAT,dhi/tsod,dhi/tsod
document,context_file,context_file,context_file
file,AGENTS.md,CLAUDE.md,CLAUDE.md
path,SharedRules,tsod/base.py,tsod/detectors.py
form,inline,inline,inline
tree_root,NaN,NaN,NaN
in_structure_section,False,True,True
language,Rust,Python,Python
language_group,other,TS/JS/Python,TS/JS/Python


## Steps 5–6 · Dependency rules and the import checker

`p04` selects candidate sentences with a broad pattern. An LLM then classified each sentence and wrote a structured rule (files covered, exempt files, forbidden folders or packages); the prompts and answers are in `data/intermediate/rule_classification/`. One author reviewed every structured rule. `p05` runs the import checker.

In [8]:
run('p04_find_rule_sentences.py'); run('p05_check_rules_and_commits.py')
sent = load('rule_candidate_sentences')
print(f'{len(sent):,} candidate sentences in {sent.repo.nunique()} projects; '
      f'{sent.category.isin(["internal_boundary","external_boundary"]).sum()} dependency rules; '
      f'{(sent.checkable & sent.category.isin(["internal_boundary","external_boundary"])).sum()} checkable')

skipped p04_find_rule_sentences.py (cached output is used)
skipped p05_check_rules_and_commits.py (cached output is used)
1,645 candidate sentences in 875 projects; 357 dependency rules; 176 checkable


**What the import checker reads**, a few example lines from different languages:

In [9]:
from import_checker import parse
for lang, line in [('py', 'from pwndbg.aglib import qemu'), ('ts', "import { Button } from '@mui/material';"),
                   ('go', '"github.com/grafana/pyroscope/pkg/og/storage"'), ('rs', 'use tokio::sync::mpsc;'),
                   ('java', 'import com.arcadedb.server.ServerPlugin;')]:
    print(f'{lang:5}', line, '->', [p[0] for p in parse(line, lang)])

py    from pwndbg.aglib import qemu -> ['pwndbg.aglib']
ts    import { Button } from '@mui/material'; -> []
go    "github.com/grafana/pyroscope/pkg/og/storage" -> ['github.com/grafana/pyroscope/pkg/og/storage']
rs    use tokio::sync::mpsc; -> ['tokio::sync::mpsc']
java  import com.arcadedb.server.ServerPlugin; -> []


## Steps 7–8 · Agent-assisted commits

A commit is *agent-assisted* if it carries a visible agent trace. We combine three kinds of traces, based on Robbes et al. (MSR 2026); the patterns are in `src/agent_patterns/`.

| trace | codes in `trace_codes` |
|---|---|
| commit message: in the dataset's AI co-author list, co-author trailer, "Generated with" line, session link | S1, S2 |
| author or committer is an agent account | S3 |
| pull request: agent-named branch, squash-merged PR with a traced commit, same merged PR as a traced commit | S4, S5, S6 |

In [10]:
run('p06_detect_agent_commits.py'); run('p07_add_branch_traces.py'); run('p08_label_commit_groups.py'); run('p09_context_file_creators.py')
from agent_traces import message_hit, ident_hit
for msg in ['Fix parser\n\nCo-authored-by: Claude <noreply@anthropic.com>', '🤖 Generated with [Claude Code](https://claude.com/claude-code)', 'Refactor utils']:
    print(repr(msg[:50]), '->', message_hit(msg))
print('copilot-swe-agent account ->', ident_hit('Copilot', '198982749+Copilot@users.noreply.github.com'))

skipped p06_detect_agent_commits.py (cached output is used)
skipped p07_add_branch_traces.py (cached output is used)
skipped p08_label_commit_groups.py (cached output is used)
skipped p09_context_file_creators.py (cached output is used)
'Fix parser\n\nCo-authored-by: Claude <noreply@anthro' -> coauthor
'🤖 Generated with [Claude Code](https://claude.com/' -> signature
'Refactor utils' -> None
copilot-swe-agent account -> Copilot


## Steps 9–11 · Review samples and clean tables

`r01`–`r04` draw the samples for the manual checks and merge the labels. `p10`/`p11` build the rule and commit tables. `p12` writes everything a person needs to read or review:

- `data/processed/`: the clean tables used by the RQ notebooks.
- `manual_review/`: one sheet per manual check (R1–R7), with the item, evidence links, the two LLM labels, the adjudicated label and the final label.

See `manual_review/README.md` for what to review and how.

In [11]:
run('p10_rule_and_commit_tables.py'); run('p11_rule_attribution.py')
subprocess.run([sys.executable, 'src/p12_export_clean_tables.py'], check=True)
pd.DataFrame([(f, len(pd.read_csv(f'{PROC}/{f}', low_memory=False))) for f in sorted(os.listdir(PROC))], columns=['data/processed/', 'rows'])

skipped p10_rule_and_commit_tables.py (cached output is used)
skipped p11_rule_attribution.py (cached output is used)


done {'R1_missing_path_claims.csv': 300, 'R2_found_path_claims.csv': 100, 'R3_rules_broken_when_written.csv': 51, 'R4_rules_that_held.csv': 40, 'R5_flagged_commits.csv': 35, 'R6_unflagged_commits.csv': 100, 'R7_agent_commit_labels.csv': 150}


,data/processed/,rows
0,commit_rule_pairs.csv,8928
1,context_files.csv,4372
2,dependency_rules.csv,184
3,exposed_commits.csv,7480
4,path_claims.csv,49388
5,projects.csv,3909
6,rule_attribution.csv,51
7,rule_candidate_sentences.csv,1645


In [12]:
pd.DataFrame([(f, len(pd.read_csv(f'{REVIEW}/{f}'))) for f in sorted(os.listdir(REVIEW)) if f.endswith('.csv')], columns=['manual_review/', 'items'])

,manual_review/,items
0,R1_missing_path_claims.csv,300
1,R2_found_path_claims.csv,100
2,R3_rules_broken_when_written.csv,51
3,R4_rules_that_held.csv,40
4,R5_flagged_commits.csv,35
5,R6_unflagged_commits.csv,100
6,R7_agent_commit_labels.csv,150
